# NCKH-UAV: Adversarial Camouflage Attack — v8 (THIẾT KẾ ĐÚNG)

**Sửa lỗi cốt tử của v5–v7:** trước đây train cả 2 model trên DOTA+DIOR gộp → chỉ khác *kiến trúc*,
mất điều kiện *khác dữ liệu*. v8 tách đúng:

- **Surrogate** = YOLOv8n train **DOTA-only** → nhóm tối ưu.
- **Target** = RT-DETR-l train **DIOR-only** → nhóm giữ kín (**khác kiến trúc + khác dữ liệu**).
- Patch tối ưu bằng DE trên surrogate/DOTA → đo **transfer** trên target/DIOR.
- Thêm **đối chứng patch ngẫu nhiên** cùng diện tích (bắt buộc).

## Tránh timeout 12h
Train 2 model trên 2 dataset (mỗi cái nhỏ hơn bản gộp). Vẫn hỗ trợ:
- `REUSE_SURROGATE` / `REUSE_TARGET`: nạp `best.pt` đã train (Add Input) → khỏi train lại.
- `STOP_AFTER_TRAINING`: chỉ train + lưu, chạy lại lần sau để tấn công.

⚠️ Nếu reuse: surrogate phải là bản **DOTA-only**, target là **DIOR-only**. KHÔNG dùng weights v5/v6 (gộp).


In [ ]:
# ── Cell 1: Env, GPU, source, weight finder ──
import os, sys, shutil, glob, time
from pathlib import Path

DOTA_INPUT = '/kaggle/input/datasets/thuyen001/dota-raw'
DIOR_INPUT = '/kaggle/input/datasets/thuyen001/dior-raw/dior_raw'
CODE_SRC   = '/kaggle/input/datasets/thuyen001/nckh-uav-code/nckh-uav-code'
WORK = Path('/kaggle/working'); DATA = WORK/'data'

dst = WORK/'src'
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(os.path.join(CODE_SRC,'src'), str(dst))
shutil.copy2(os.path.join(CODE_SRC,'configs/base_config.yaml'), str(WORK/'base_config.yaml'))
sys.path.insert(0, str(WORK))

import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

def find_weights(keywords, prefer='best'):
    kws=[k.lower() for k in keywords]; hits=[]
    for pt in glob.glob('/kaggle/input/**/*.pt', recursive=True):
        low=pt.lower(); nm=os.path.basename(low)
        if nm in ('yolov8n.pt','rtdetr-l.pt') and 'runs' not in low and 'weights' not in low:
            continue
        if any(k in low for k in kws): hits.append(pt)
    hits.sort(key=lambda p:(0 if prefer in os.path.basename(p).lower() else 1, p))
    return hits

print('\n── .pt found in /kaggle/input ──')
for p in glob.glob('/kaggle/input/**/*.pt', recursive=True): print('  ', p)


In [ ]:
# ── Cell 2: Install ──
!pip -q install ultralytics opencv-python-headless pyyaml tqdm


In [ ]:
# ══════════════════════════════════════════════════════════
#  Cell 3: RUN CONFIG
# ══════════════════════════════════════════════════════════
REUSE_SURROGATE = False   # True = nạp YOLOv8n DOTA-only best.pt (Add Input)
REUSE_TARGET    = False   # True = nạp RT-DETR DIOR-only best.pt

SURR_PT_OVERRIDE   = None
TARGET_PT_OVERRIDE = None

YOLO_EPOCHS   = 50        # surrogate (DOTA)
RTDETR_EPOCHS = 30        # target (DIOR) — giảm để tiết kiệm
RTDETR_IMGSZ  = 640
RTDETR_BATCH  = 8
YOLO_IMGSZ    = 1024
YOLO_BATCH    = 8
USE_CACHE     = True

STOP_AFTER_TRAINING = False   # True = chỉ train + lưu, bỏ qua tấn công

SURR_PT = TARGET_PT = None
if REUSE_SURROGATE:
    SURR_PT = SURR_PT_OVERRIDE or (find_weights(['surrogate','dota','yolov8n']) or [None])[0]
    assert SURR_PT, 'REUSE_SURROGATE=True nhưng không thấy YOLOv8n(DOTA) best.pt trong /kaggle/input.'
    print('Surrogate (reuse):', SURR_PT)
if REUSE_TARGET:
    TARGET_PT = TARGET_PT_OVERRIDE or (find_weights(['rtdetr','dior','target','detr']) or [None])[0]
    assert TARGET_PT, 'REUSE_TARGET=True nhưng không thấy RT-DETR(DIOR) best.pt trong /kaggle/input.'
    print('Target (reuse):', TARGET_PT)

print(f'\nPlan: surrogate={"REUSE" if REUSE_SURROGATE else f"TRAIN {YOLO_EPOCHS}ep DOTA"} | '
      f'target={"REUSE" if REUSE_TARGET else f"TRAIN {RTDETR_EPOCHS}ep DIOR"} | '
      f'stop_after_training={STOP_AFTER_TRAINING}')


In [ ]:
# ── Cell 4: Symlink raw DOTA / DIOR ──
dota_img_src = os.path.join(DOTA_INPUT,'images'); dota_lbl_src = os.path.join(DOTA_INPUT,'labelTxt')
dota_dst = DATA/'DOTA'/'raw'; dota_dst.mkdir(parents=True, exist_ok=True)
for n,s in [('images',dota_img_src),('labelTxt',dota_lbl_src)]:
    lk=dota_dst/n
    if not lk.exists(): os.symlink(s,str(lk))
dior_dst = DATA/'DIOR'/'raw'; dior_dst.mkdir(parents=True, exist_ok=True)
for f in ['JPEGImages','Annotations']:
    lk=dior_dst/f; s=os.path.join(DIOR_INPUT,f)
    if os.path.isdir(s) and not lk.exists(): os.symlink(s,str(lk))
print('DOTA imgs:', len(list(Path(dota_img_src).glob('*.png'))))
print('DIOR imgs:', len(glob.glob(str(dior_dst/'JPEGImages'/'*.jpg'))))


In [ ]:
# ── Cell 5: Process DOTA (slice) + DIOR (XML→YOLO) ──
import cv2, numpy as np
from PIL import Image
from tqdm import tqdm
Image.MAX_IMAGE_PIXELS = None

DOTA_CLASSES={'plane':0,'small-vehicle':1,'large-vehicle':1}; SLICE,OVL=1024,0.2
d_img=DATA/'DOTA'/'processed'/'images'; d_img.mkdir(parents=True,exist_ok=True)
d_lbl=DATA/'DOTA'/'processed'/'labels'; d_lbl.mkdir(parents=True,exist_ok=True)
raw=sorted(glob.glob(str(DATA/'DOTA'/'raw'/'images'/'*.png')))

def parse_dota(lp):
    a=[]
    for line in open(lp):
        q=line.strip().split()
        if len(q)<9 or q[8] not in DOTA_CLASSES: continue
        c=list(map(float,q[:8])); xs,ys=c[0::2],c[1::2]
        a.append((min(xs),min(ys),max(xs),max(ys),DOTA_CLASSES[q[8]]))
    return a

ns=0
for ip in tqdm(raw,desc='DOTA'):
    stem=Path(ip).stem; lp=str(DATA/'DOTA'/'raw'/'labelTxt'/f'{stem}.txt')
    if not os.path.isfile(lp): continue
    anns=parse_dota(lp)
    if not anns: continue
    try:
        im=np.array(Image.open(ip))
        im=cv2.cvtColor(im,cv2.COLOR_GRAY2BGR) if im.ndim==2 else (cv2.cvtColor(im,cv2.COLOR_RGBA2BGR) if im.shape[2]==4 else cv2.cvtColor(im,cv2.COLOR_RGB2BGR))
    except Exception: continue
    h,w=im.shape[:2]; st=int(SLICE*(1-OVL))
    for y0 in range(0,h,st):
        for x0 in range(0,w,st):
            x1,y1=min(x0+SLICE,w),min(y0+SLICE,h); x0a,y0a=max(0,x1-SLICE),max(0,y1-SLICE)
            sa=[]
            for (ax1,ay1,ax2,ay2,cid) in anns:
                ix1,iy1=max(ax1,x0a),max(ay1,y0a); ix2,iy2=min(ax2,x1),min(ay2,y1)
                iw,ih=ix2-ix1,iy2-iy1
                if iw>10 and ih>10 and (iw*ih)/max((ax2-ax1)*(ay2-ay1),1)>0.3:
                    sw,sh=x1-x0a,y1-y0a
                    sa.append(f'{cid} {((ix1-x0a)+(ix2-x0a))/2/sw:.6f} {((iy1-y0a)+(iy2-y0a))/2/sh:.6f} {iw/sw:.6f} {ih/sh:.6f}')
            if not sa: continue
            nm=f'{stem}_{x0a}_{y0a}'
            cv2.imwrite(str(d_img/f'{nm}.jpg'), im[y0a:y1,x0a:x1])
            open(str(d_lbl/f'{nm}.txt'),'w').write('\n'.join(sa)+'\n'); ns+=1
print('DOTA slices:', ns)

from src.data_prep.process_dior import main as proc_dior
o_img=DATA/'DIOR'/'processed'/'images'; o_img.mkdir(parents=True,exist_ok=True)
o_lbl=DATA/'DIOR'/'processed'/'labels'; o_lbl.mkdir(parents=True,exist_ok=True)
nd=proc_dior(input_img_dir=str(DATA/'DIOR'/'raw'/'JPEGImages'),
             input_xml_dir=str(DATA/'DIOR'/'raw'/'Annotations'),
             out_img_dir=str(o_img), out_lbl_dir=str(o_lbl), use_symlink=True)
print('DIOR imgs:', nd)


In [ ]:
# ── Cell 6: Build TWO separate datasets (yolo_dota, yolo_dior) ──
import random
def collect(idir,ldir):
    ps=[]
    for p in Path(idir).iterdir():
        if p.suffix.lower() not in ('.jpg','.jpeg','.png','.bmp'): continue
        lp=Path(ldir)/(p.stem+'.txt')
        if lp.exists() and lp.stat().st_size>5: ps.append((str(p),str(lp)))
    return ps

def build(pairs, root, sr=0.8, seed=42):
    for s in ['train/images','train/labels','val/images','val/labels']:
        (root/s).mkdir(parents=True, exist_ok=True)
    random.Random(seed).shuffle(pairs)
    k=int(len(pairs)*sr)
    for subset,ss in [('train',pairs[:k]),('val',pairs[k:])]:
        for ip,lp in ss:
            di=root/subset/'images'/Path(ip).name; dl=root/subset/'labels'/Path(lp).name
            if not di.exists(): os.symlink(ip,str(di))
            if not dl.exists(): os.symlink(lp,str(dl))
    (root/'data.yaml').write_text(f'path: {root}\ntrain: train/images\nval: val/images\nnc: 2\nnames: ["aircraft","vehicle"]\n')
    return k, len(pairs)-k

DOTA_ROOT=DATA/'yolo_dota'; DIOR_ROOT=DATA/'yolo_dior'
ta,va=build(collect(d_img,d_lbl), DOTA_ROOT)
ti,vi=build(collect(o_img,o_lbl), DIOR_ROOT)
DOTA_YAML=str(DOTA_ROOT/'data.yaml'); DIOR_YAML=str(DIOR_ROOT/'data.yaml')
DOTA_VI,DOTA_VL=str(DOTA_ROOT/'val/images'),str(DOTA_ROOT/'val/labels')
DIOR_VI,DIOR_VL=str(DIOR_ROOT/'val/images'),str(DIOR_ROOT/'val/labels')
print(f'DOTA(surrogate) train {ta}/val {va} | DIOR(target) train {ti}/val {vi}')


In [ ]:
# ── Cell 7: Surrogate YOLOv8n on DOTA-only ──
from ultralytics import YOLO
if REUSE_SURROGATE:
    assert os.path.isfile(SURR_PT), SURR_PT
    print('✓ reuse surrogate:', SURR_PT)
else:
    print(f'Train YOLOv8n on DOTA {YOLO_EPOCHS}ep imgsz={YOLO_IMGSZ} batch={YOLO_BATCH}')
    YOLO('yolov8n.pt').train(data=DOTA_YAML, epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ, batch=YOLO_BATCH,
        device='0,1', project=str(WORK/'runs/detect'), name='surrogate_yolov8n_dota',
        exist_ok=True, patience=15, cache=USE_CACHE, save=True, plots=True, verbose=True)
    SURR_PT=str(WORK/'runs/detect/surrogate_yolov8n_dota/weights/best.pt')
print('SURR_PT =', SURR_PT)


In [ ]:
# ── Cell 8: Target RT-DETR-l on DIOR-only ──
from ultralytics import YOLO
if REUSE_TARGET:
    assert os.path.isfile(TARGET_PT), TARGET_PT
    print('✓ reuse target:', TARGET_PT)
else:
    print(f'Train RT-DETR-l on DIOR {RTDETR_EPOCHS}ep imgsz={RTDETR_IMGSZ} batch={RTDETR_BATCH}')
    YOLO('rtdetr-l.pt').train(data=DIOR_YAML, epochs=RTDETR_EPOCHS, imgsz=RTDETR_IMGSZ, batch=RTDETR_BATCH,
        device='0,1', project=str(WORK/'runs/detect'), name='target_rtdetr_dior',
        exist_ok=True, patience=15, cache=USE_CACHE, save=True, plots=True, verbose=True)
    TARGET_PT=str(WORK/'runs/detect/target_rtdetr_dior/weights/best.pt')
print('TARGET_PT =', TARGET_PT)
if STOP_AFTER_TRAINING:
    print('\n⏹ STOP_AFTER_TRAINING=True — models saved. Re-run with REUSE_*=True to attack.')


In [ ]:
# ── Cell 9: ★ Attack (DE + position-opt) + random control + transfer DOTA→DIOR ★ ──
import numpy as np
from src.attack.de_attack_real import DEAttackReal, plot_convergence, plot_patch, save_attack_examples

ATTACK_DIR=WORK/'attack_results'; ATTACK_DIR.mkdir(exist_ok=True)
SEED=42
ATTACK_CONFIGS=[
    dict(name='random_single',   patch_size=32, num_patches=1, optimize_positions=False, random=True),
    dict(name='baseline_center', patch_size=32, num_patches=1, optimize_positions=False, random=False),
    dict(name='opt_position',    patch_size=32, num_patches=1, optimize_positions=True,  random=False),
    dict(name='multi_patch_opt', patch_size=20, num_patches=5, optimize_positions=True,  random=False),
]
POP,GEN,F,CR=20,50,0.8,0.7
W_OBJ,W_FP,W_TV,W_NPS=1.0,0.3,0.01,0.001
N_ATTACK,N_HELD=30,50

if STOP_AFTER_TRAINING:
    print('Skipped (STOP_AFTER_TRAINING).')
else:
    atk_s=DEAttackReal(str(SURR_PT),   device='0', conf_threshold=0.25, iou_threshold=0.5, verbose=True)
    atk_t=DEAttackReal(str(TARGET_PT), device='0', conf_threshold=0.25, iou_threshold=0.5, verbose=True)
    results={}
    for c in ATTACK_CONFIGS:
        name=c['name']; ps=c['patch_size']; npa=c['num_patches']; rnd=c['random']
        print('\n'+'#'*70+f'\n# {name} | patch={ps} n={npa} opt_pos={c["optimize_positions"]} random={rnd}\n'+'#'*70)
        if rnd:
            np.random.seed(SEED)
            best_patch=np.random.randint(0,256,(ps,ps,3),dtype=np.uint8); pos=None; hist=None
        else:
            atk_s.prepare_attack_set(DOTA_VI,DOTA_VL,n_images=N_ATTACK,seed=SEED)
            best_patch,hist=atk_s.run_de(patch_size=ps,num_patches=npa,optimize_positions=c['optimize_positions'],
                pop_size=POP,max_gen=GEN,F=F,CR=CR,w_obj=W_OBJ,w_fp=W_FP,w_tv=W_TV,w_nps=W_NPS)
            pos=atk_s.last_positions
        cdir=ATTACK_DIR/name; cdir.mkdir(exist_ok=True)
        np.save(str(cdir/'best_patch.npy'),best_patch)
        if pos is not None: np.save(str(cdir/'positions.npy'),pos)
        if hist is not None: plot_convergence(hist,str(cdir/'convergence.png'))
        plot_patch(best_patch,str(cdir/'patch.png'))
        # surrogate held-out (DOTA)
        m_s=atk_s.evaluate_patch(best_patch,positions=pos,img_dir=DOTA_VI,lbl_dir=DOTA_VL,n_images=N_HELD,num_patches=npa,seed=99)
        atk_s.prepare_attack_set(DOTA_VI,DOTA_VL,n_images=6,seed=SEED)
        save_attack_examples(atk_s,best_patch,num_patches=npa,positions=pos,save_dir=str(cdir/'examples_surrogate_dota'),n=6)
        # transfer → target (DIOR)
        m_t=atk_t.evaluate_patch(best_patch,positions=pos,img_dir=DIOR_VI,lbl_dir=DIOR_VL,n_images=N_HELD,num_patches=npa,seed=99)
        atk_t.prepare_attack_set(DIOR_VI,DIOR_VL,n_images=6,seed=SEED)
        save_attack_examples(atk_t,best_patch,num_patches=npa,positions=pos,save_dir=str(cdir/'examples_target_dior'),n=6)
        results[name]=dict(config=c,surrogate=m_s,target=m_t,positions=None if pos is None else pos.tolist())
        print(f'  [{name}] surrogate(DOTA) ASR={m_s["ASR"]:.3f} | transfer(DIOR) ASR={m_t["ASR"]:.3f}')
    np.save(str(ATTACK_DIR/'results.npy'),results,allow_pickle=True)
    import json as _json
    _json.dump(results, open(ATTACK_DIR/'results.json','w'), indent=2)
    print('\n✓ attack done')


In [ ]:
# ── Cell 10: Summary table — transfer DOTA→DIOR + random control ──
if STOP_AFTER_TRAINING:
    print('Skipped (STOP_AFTER_TRAINING).')
else:
    hdr=f'{"Config":>16s} | {"surr ASR":>9s} {"AORR":>6s} | {"TRANSFER ASR":>12s} {"AORR":>6s} | {"transfer%":>9s}'
    print('='*len(hdr)); print(hdr); print('-'*len(hdr))
    rand_t=results.get('random_single',{}).get('target',{}).get('ASR',None)
    csv=['config,surr_ASR,surr_AORR,transfer_ASR,transfer_AORR,transfer_pct,is_random']
    for name,r in results.items():
        s,t=r['surrogate'],r['target']; tr=(t['ASR']/s['ASR']*100) if s['ASR']>0 else 0.0
        tag='  (control)' if r['config'].get('random') else ''
        print(f'{name:>16s} | {s["ASR"]:>9.3f} {s["AORR"]:>6.3f} | {t["ASR"]:>12.3f} {t["AORR"]:>6.3f} | {tr:>8.1f}%{tag}')
        csv.append(f'{name},{s["ASR"]},{s["AORR"]},{t["ASR"]},{t["AORR"]},{tr},{int(bool(r["config"].get("random")))}')
    print('-'*len(hdr))
    open(ATTACK_DIR/'summary.csv','w').write('\n'.join(csv)+'\n')
    if rand_t is not None:
        print(f'\nĐối chứng: random transfer ASR = {rand_t:.3f}. Phương pháp PHẢI vượt rõ mức này.')
    print('Chỉ số CHÍNH = TRANSFER ASR (model giữ kín DIOR, khác kiến trúc + dữ liệu).')


In [ ]:
# ── Cell 11: Display patches & transfer examples ──
if not STOP_AFTER_TRAINING:
    import matplotlib.pyplot as plt, matplotlib.image as mpimg
    names=list(results.keys())
    for name in names:
        cdir=ATTACK_DIR/name
        imgs=[str(cdir/'patch.png')]
        ex=sorted(glob.glob(str(cdir/'examples_target_dior'/'*.jpg')))
        if ex: imgs.append(ex[0])
        fig,ax=plt.subplots(1,len(imgs),figsize=(6*len(imgs),4))
        if len(imgs)==1: ax=[ax]
        for a,im in zip(ax,imgs): a.imshow(mpimg.imread(im)); a.axis('off')
        ax[0].set_title(f'{name} — patch')
        if len(imgs)>1: ax[1].set_title(f'{name} — transfer on RT-DETR/DIOR')
        plt.tight_layout(); plt.show()
else:
    print('Skipped (STOP_AFTER_TRAINING).')
